# Core Base Module - Entity and Registry

This notebook demonstrates how to use the `core.base` module, which provides:
- **BaseEntity**: Base class for SQLAlchemy ORM models
- **Document**: Base class for document database models (MongoDB, etc.)
- **EntityRegistry**: Automatic entity discovery and table creation

## Table of Contents
1. [Creating Entities](#creating-entities)
2. [Entity Registry](#entity-registry)
3. [Automatic Table Creation](#automatic-table-creation)
4. [Document Models](#document-models)

In [7]:
# Import required modules
import os, sys 
sys.path.append(os.path.abspath(".."))

from core.base.entity import BaseEntity, Document
from core.base.registry import EntityRegistry
from core.database import PostgresDatabaseManagerImpl
from sqlalchemy import Column, String, Integer, ForeignKey
from datetime import datetime
from typing import Optional

print("✅ Imports successful!")

✅ Imports successful!


## 1. Creating Entities

Entities are SQLAlchemy ORM models that inherit from `BaseEntity`. They automatically get:
- `id`: Primary key
- `created_at`: Timestamp
- `updated_at`: Timestamp
- Automatic table name generation (class name → lowercase)

**Note**: While `BaseEntity` provides a `__tablename__()` classmethod, it's recommended to explicitly set `__tablename__` as a class attribute when using foreign keys to ensure SQLAlchemy can properly resolve table names during table creation.

In [14]:
# Example 1: Simple Entity
class User(BaseEntity):
    """User entity example"""
    __tablename__ = "user"  # Explicitly set table name as class attribute
    
    name = Column(String(100), nullable=False)
    email = Column(String(255), unique=True, nullable=False)
    age = Column(Integer)

# Create an instance
user = User(name="John Doe", email="john@example.com", age=30)
print(f"Created user: {user.name}, email: {user.email}")
print(f"Table name: {User.__tablename__}")
print(f"Has id: {hasattr(user, 'id')}")
print(f"Has created_at: {hasattr(user, 'created_at')}")

Created user: John Doe, email: john@example.com
Table name: user
Has id: True
Has created_at: True


/var/folders/04/kxzm_nrn6w9fwt4f_qnyr1m00000gn/T/ipykernel_7453/2091353867.py:2: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.User, and will be replaced in the string-lookup table.
  class User(BaseEntity):


In [15]:
# Example 2: Entity with Foreign Key Relationship
class Post(BaseEntity):
    """Post entity with foreign key to User"""
    __tablename__ = "post"  # Explicitly set table name as class attribute
    
    title = Column(String(200), nullable=False)
    content = Column(String(5000))
    user_id = Column(Integer, ForeignKey('user.id'))  # Foreign key to User

# Create instances
post = Post(title="My First Post", content="This is the content", user_id=1)
print(f"Created post: {post.title}")
print(f"Table name: {Post.__tablename__}")

Created post: My First Post
Table name: post


/var/folders/04/kxzm_nrn6w9fwt4f_qnyr1m00000gn/T/ipykernel_7453/521343114.py:2: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Post, and will be replaced in the string-lookup table.
  class Post(BaseEntity):


## 2. Entity Registry

The `EntityRegistry` automatically discovers and manages entities, handling dependency ordering for table creation.

In [16]:
# Register entities manually
EntityRegistry.register(User)
EntityRegistry.register(Post)

# Get all registered entities
entities = EntityRegistry.get_all_entities()
print(f"Registered entities: {[e.__name__ for e in entities]}")

# Get a specific entity
user_entity = EntityRegistry.get_entity("User")
print(f"Retrieved entity: {user_entity.__name__}")

2026-01-18 07:45:43.200 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for User: set()
2026-01-18 07:45:43.201 | DEBUG    | core.base.registry:register:25 - Registered entity: User
2026-01-18 07:45:43.202 | DEBUG    | core.base.registry:_analyze_dependencies:49 - Found string FK dependency: Post -> User
2026-01-18 07:45:43.203 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for Post: {'User'}
2026-01-18 07:45:43.203 | DEBUG    | core.base.registry:register:25 - Registered entity: Post
2026-01-18 07:45:43.204 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for User: set()
2026-01-18 07:45:43.204 | DEBUG    | core.base.registry:_analyze_dependencies:49 - Found string FK dependency: Post -> User
2026-01-18 07:45:43.205 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for Post: {'User'}
2026-01-18 07:45:43.205 | INFO     | core.base.registry:_topological_sort:106 - All detected depen

Registered entities: ['User', 'Post']
Retrieved entity: User


In [17]:
# Example: Automatic entity discovery from a package
# This would discover all entities in the 'entities' package
# EntityRegistry.discover_entities('entities')

# For demonstration, we'll show manual registration
print("Entity Registry Features:")
print(f"- Total registered: {len(EntityRegistry._entities)}")
print(f"- Entities: {list(EntityRegistry._entities.keys())}")

Entity Registry Features:
- Total registered: 2
- Entities: ['User', 'Post']


## 3. Automatic Table Creation

The registry can automatically create database tables in the correct order (respecting foreign key dependencies).

In [18]:
# Example: Create tables using database manager
# Note: This requires a database connection

# Setup SQLite in-memory database for demonstration
db_manager = PostgresDatabaseManagerImpl(engine_info="sqlite:///:memory:")
db_manager.connect()

# Create tables for all registered entities
success = EntityRegistry.create_tables(db_manager, drop_first=False)
print(f"Tables created: {success}")

# Verify tables exist
if db_manager.is_connected():
    with db_manager.get_session() as session:
        # Tables should now exist
        print("✅ Database tables created successfully!")
        print(f"Available entities: {[e.__name__ for e in EntityRegistry.get_all_entities()]}")

db_manager.close()

2026-01-18 07:45:45.925 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for User: set()
2026-01-18 07:45:45.926 | DEBUG    | core.base.registry:_analyze_dependencies:49 - Found string FK dependency: Post -> User
2026-01-18 07:45:45.927 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for Post: {'User'}
2026-01-18 07:45:45.927 | INFO     | core.base.registry:_topological_sort:106 - All detected dependencies: {'User': set(), 'Post': {'User'}}
2026-01-18 07:45:45.928 | INFO     | core.base.registry:_topological_sort:115 - Dependency graph: {'User': set(), 'Post': {'User'}}
2026-01-18 07:45:45.929 | INFO     | core.base.registry:get_all_entities:150 - Entity creation order: ['User', 'Post']
2026-01-18 07:45:45.936 | INFO     | core.database.postgres:create_tables:199 - Tables created successfully
2026-01-18 07:45:45.937 | DEBUG    | core.base.registry:_analyze_dependencies:62 - FK Dependencies for User: set()
2026-01-18 07:45:45.937 | DE

Tables created: True
✅ Database tables created successfully!
Available entities: ['User', 'Post']


## 4. Document Models

For document databases (MongoDB, etc.), use the `Document` base class instead of `BaseEntity`.

In [19]:
# Example: Document model for MongoDB
class Article(Document):
    """Article document for MongoDB"""
    title: str
    content: str
    author: str
    tags: list[str] = []

# Create document instance
article = Article(
    title="Getting Started with Python",
    content="Python is a great language...",
    author="Jane Smith",
    tags=["python", "tutorial"]
)

# Get collection name (auto-generated from class name)
collection_name = Article.get_collection_name()
print(f"Collection name: {collection_name}")

# Convert to dictionary
article_dict = article.to_dict()
print(f"Article dict: {article_dict}")

Collection name: articles
Article dict: {'title': 'Getting Started with Python', 'content': 'Python is a great language...', 'author': 'Jane Smith', 'tags': ['python', 'tutorial']}


## Summary

**BaseEntity** (SQL databases):
- Inherit from `BaseEntity` for SQLAlchemy ORM models
- Automatic table name generation
- Built-in `id`, `created_at`, `updated_at` fields
- Use with PostgreSQL, MySQL, SQLite

**Document** (Document databases):
- Inherit from `Document` for MongoDB/document stores
- Automatic collection name generation
- Pydantic-based validation
- Use with MongoDB, CouchDB

**EntityRegistry**:
- Automatic entity discovery
- Dependency-aware table creation
- Topological sorting for foreign keys